# SentinelRUL — EDA on NASA C-MAPSS FD001
Exploratory look at the training data before modelling.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from src.data.loader import load_train, load_test, load_rul
from src.data.preprocess import SENSOR_COLS, add_rul_labels, prepare_train
from src.data.windows import WINDOW_SIZE, make_windows

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_palette('tab10')

In [ ]:
train_raw = load_train()
test_raw  = load_test()
rul_df    = load_rul()

print(f'train shape : {train_raw.shape}')
print(f'test  shape : {test_raw.shape}')
print(f'engines in train : {train_raw.engine_id.nunique()}')
print(f'engines in test  : {test_raw.engine_id.nunique()}')
train_raw.head(3)

In [ ]:
train_df = add_rul_labels(train_raw.copy())

engine_life = train_df.groupby('engine_id')['cycle'].max()

print(f'engines            : {train_df.engine_id.nunique()}')
print(f'total rows         : {len(train_df):,}')
print(f'RUL range          : {int(train_df.rul.min())} to {int(train_df.rul.max())}')
print(f'avg lifetime       : {engine_life.mean():.1f} cycles')
print(f'shortest engine    : {engine_life.min()} cycles  (engine {engine_life.idxmin()})')
print(f'longest engine     : {engine_life.max()} cycles  (engine {engine_life.idxmax()})')

train_df[['engine_id', 'cycle', 'rul']].groupby('engine_id').last().describe()

In [ ]:
sample_engines = sorted(train_raw['engine_id'].unique())[::20][:5]

n_sensors = len(SENSOR_COLS)
ncols = 4
nrows = (n_sensors + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows), sharey=False)
axes_flat = axes.flatten()

for ax_idx, sensor in enumerate(SENSOR_COLS):
    ax = axes_flat[ax_idx]
    for eid in sample_engines:
        eng = train_raw[train_raw['engine_id'] == eid].sort_values('cycle')
        max_c = eng['cycle'].max()
        ax.plot(eng['cycle'] / max_c, eng[sensor], alpha=0.7, linewidth=1, label=f'engine {eid}')
    ax.set_title(sensor, fontsize=10)
    ax.set_xlabel('normalised cycle', fontsize=8)

for ax in axes_flat[n_sensors:]:
    ax.set_visible(False)

axes_flat[0].legend(fontsize=7, loc='upper right')
fig.suptitle('Sensor degradation trajectories (5 sample engines)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr = train_raw[SENSOR_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Pairwise sensor correlations', fontsize=13)
plt.tight_layout()
plt.show()